# 02 – Character Anime Works

Esplorazione e data cleaning del dataset `character_anime_works.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `anime_mal_id` | ID univoco dell'anime su MyAnimeList |
| `character_mal_id` | ID univoco del personaggio su MyAnimeList |
| `character_name` | Nome del personaggio |
| `role` | Ruolo del personaggio nell'anime |

## 1. Import e caricamento dati
Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [9]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_works = pd.read_csv('../datasets/character_anime_works.csv')
print(f'Shape: {df_works.shape}')
df_works.info()
df_works.head()

Shape: (236816, 4)
<class 'pandas.DataFrame'>
RangeIndex: 236816 entries, 0 to 236815
Data columns (total 4 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   anime_mal_id      236816 non-null  int64
 1   character_mal_id  236816 non-null  int64
 2   character_name    236816 non-null  str  
 3   role              236816 non-null  str  
dtypes: int64(2), str(2)
memory usage: 7.2 MB


,anime_mal_id,character_mal_id,character_name,role
0,2928,5781,Atoli,Main
1,2928,33,Haseo,Main
2,2928,32,Ovan,Main
3,2928,34,Shino,Main
4,2928,5785,Aina,Supporting


Il dataset contiene **236.816 righe** e **4 colonne**. Tutte le colonne sono complete: **nessun valore nullo** in nessuna delle 4 colonne. I tipi di dati sono adeguati: `int64` per gli ID e `str` per le colonne testuali.

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [10]:
n_originale = len(df_works)

mask_dup = df_works.duplicated(keep=False)       # tutte le occorrenze coinvolte
n_righe_coinvolte = mask_dup.sum()                    # righe totali che hanno almeno un duplicato
n_gruppi = df_works[mask_dup].duplicated(keep='first').sum()  # occorrenze extra (da rimuovere)
n_tenute = n_righe_coinvolte - n_gruppi               # prime occorrenze mantenute

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_works.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_works):,}')

Righe totali coinvolte in duplicazioni : 0
  → prime occorrenze mantenute         : 0
  → occorrenze extra rimosse           : 0

Righe prima della rimozione : 236,816
Righe dopo la rimozione     : 236,816


Nessun duplicato esatto trovato. Tutte le 236.816 righe sono già uniche. Il dataset rimane invariato.

Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `anime_mal_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria di `details.csv`.

Poiché il dataset padre pulito non è ancora disponibile, utilizziamo `analyze` per verificare la presenza di valori nulli e avere una panoramica della colonna.

I valori duplicati sono **attesi**: ogni anime è associato a più personaggi.

In [11]:
analyze(df_works['anime_mal_id'])


  Nome serie                     anime_mal_id
  dtype                          int64
  Dimensione                     236,816
  Range indice                   0 … 236815
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   236,816
  Valori non nulli               236,816  (100.00%)
  Null / NaN                     0  (0.00%)
  Zeri                           0  (0.00%)
  Positivi                       236,816   (100.00%)
  Negativi                       0   (0.00%)
  Valori unici                   15,284  (6.45%)

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            1
  Max                            62,584
  Range                          62,583
  Media                          23,725.50
  Mediana                        21,013.00
  Moda/e                         235
  Dev. standar

**Osservazioni:**
- **Nessun valore nullo**: tutti i 236.816 record hanno un ID anime valido.
- **Duplicati attesi**: ogni anime è associato a più personaggi.

Nessuna pulizia è necessaria.

### 2.2 `character_mal_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria di `characters.csv`. Per una chiave esterna le statistiche descrittive non hanno significato interpretativo.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID presente qui deve esistere in `characters_clean.csv`.

Usiamo quindi `check_fk` al posto di `analyze`, che effettua entrambi i controlli.

I valori duplicati sono **attesi**: lo stesso personaggio può apparire in più anime (es. crossover, sequel).

In [12]:
df_characters = pd.read_csv('../datasets_cleaned/characters_clean.csv')

mask_orphan = check_fk(
    df_works['character_mal_id'],
    df_characters['character_mal_id'],
    child_df=df_works
)

print(f'Null in character_mal_id  : {df_works["character_mal_id"].isna().sum()}')
print(f'Duplicati in character_mal_id (attesi) : {df_works["character_mal_id"].duplicated().sum():,}')


  Colonna FK  (tabella figlia)        character_mal_id
  Colonna PK  (tabella padre)         character_mal_id
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       236,816
  Valori null nella FK                0  (0.00%)
  Valori non-null nella FK            236816  (100.00%)
  Valori unici nella PK padre         209,506

  ✓  Righe con FK valida              236814  (100.00%)
  ✗  Righe orfane (FK non in PK)      2  (0.00%)
     → ID orfani unici                2

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

2 riga/e (0.00%) violano l'integrità referenziale.

ID orfani
────────────────────────────────────────────────────────────────────────────────

  [np.int64(281878), np.int64(281879)]

Campione righe orfane (prime 2)
─────────────────────────────

**Osservazioni:**
- **Nessun valore nullo**: tutti i 236.816 record hanno un ID personaggio di riferimento.
- **Integrità referenziale**: 2 righe orfane rilevate (`character_mal_id` 281878 e 281879, entrambe relative all'anime 3009). I corrispondenti personaggi non sono presenti in `characters_clean.csv`. Le righe vanno rimosse.

In [17]:
if mask_orphan.any():
    n_orfane = mask_orphan.sum()
    df_works = df_works[~mask_orphan].reset_index(drop=True)
    print(f'Righe orfane rimosse : {n_orfane}')
    print(f'Righe rimanenti      : {len(df_works):,}')
else:
    print('Nessuna riga orfana da rimuovere.')

Righe orfane rimosse : 2
Righe rimanenti      : 236,814


### 2.3 `character_name`

Colonna che contiene il nome del personaggio. Ci interessa verificare la presenza di valori nulli, duplicati e caratteri anomali. Per questo utilizziamo `analyze`.

I duplicati sono **attesi** in quanto personaggi diversi possono condividere lo stesso nome e lo stesso personaggio appare in più righe (una per ogni anime in cui compare).

In [18]:
analyze(df_works['character_name'])


  Nome serie                     character_name
  dtype                          str
  Dimensione                     236,814
  Range indice                   0 … 236813
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   236,814
  Valori non nulli               236,814  (100.00%)
  Null / NaN                     0  (0.00%)
  Stringhe vuote                 0
  Valori unici                   114,584  (48.39%)
  Valori duplicati               122,230

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  1  caratteri
  Lunghezza max                  55  caratteri
  Lunghezza media                11.23  caratteri
  Lunghezza mediana              12.0000  caratteri
  Dev. standard lunghezza        4.90
  IQR lunghezza                  8.0000

Valori di lunghezza estrema
──

**Osservazioni:**
- **Nessun valore nullo**: tutti i 236.816 record hanno un nome personaggio.
- **Duplicati attesi**: personaggi diversi possono condividere lo stesso nome e lo stesso personaggio appare più volte.

Nessuna pulizia è necessaria.

### 2.4 `role`

Colonna che indica il ruolo del personaggio nell'anime. Utilizziamo `analyze` per verificare che non ci siano valori nulli o anomalie.

In [19]:
analyze(df_works['role'])


  Nome serie                     role
  dtype                          str
  Dimensione                     236,814
  Range indice                   0 … 236813
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   236,814
  Valori non nulli               236,814  (100.00%)
  Null / NaN                     0  (0.00%)
  Stringhe vuote                 0
  Valori unici                   2  (0.00%)
  Valori duplicati               236,812

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  4  caratteri
  Lunghezza max                  10  caratteri
  Lunghezza media                8.72  caratteri
  Lunghezza mediana              10.0000  caratteri
  Dev. standard lunghezza        2.46
  IQR lunghezza                  0.0000

Valori di lunghezza estrema
────────────────────

**Osservazioni:**
- **Nessun valore nullo**: tutti i 236.816 record hanno un ruolo definito.
- **Solo 2 valori distinti**: `Main` e `Supporting`.
- La distribuzione (~78.6% `Supporting`, ~21.4% `Main`) è coerente in quanto i personaggi principali sono sempre una minoranza in confronto ai personaggi di supporto. 

Nessuna pulizia è necessaria.

### 2.5 Chiave primaria `(anime_mal_id, character_mal_id)`

Questa tabella è una relazione molti-a-molti tra anime e personaggi. La chiave primaria è la coppia `(anime_mal_id, character_mal_id)`. Verifichiamo che sia univoca.

In [15]:
n_pk_dup = df_works.duplicated(subset=['anime_mal_id', 'character_mal_id'], keep=False).sum()
print(f'Duplicati su (anime_mal_id, character_mal_id): {n_pk_dup}')

if n_pk_dup > 0:
    print('→ Rimozione duplicati sulla chiave primaria...')
    df_works.drop_duplicates(subset=['anime_mal_id', 'character_mal_id'], keep='first', inplace=True)
    print(f'Righe dopo la rimozione: {len(df_works):,}')
else:
    print('→ Chiave primaria già univoca, nessuna operazione richiesta.')

Duplicati su (anime_mal_id, character_mal_id): 0
→ Chiave primaria già univoca, nessuna operazione richiesta.


**Osservazioni:**
La coppia `(anime_mal_id, character_mal_id)` è **univoca** per tutte le righe. La struttura della tabella è corretta.

## 3. Riepilogo e Salvataggio
Le operazioni di pulizia sono state effettuate colonna per colonna nella sezione 2. In questa sezione riepiloghiamo il risultato ed effettuiamo il salvataggio del dataset finale.

In [20]:
print('Riepilogo Dataset Pulito')
print(f'Righe originali      : {n_originale:>10,}')
print(f'Righe dopo cleaning  : {len(df_works):>10,}')
print(f'Righe rimosse totali : {n_originale - len(df_works):>10,}')
print()
print('Dtypes finali:')
print(df_works.dtypes)
print()
print('Valori mancanti residui:')
print(df_works.isnull().sum())
print()
df_works.to_csv('../datasets_cleaned/character_anime_works_clean.csv', index=False)
print('Salvato: datasets_cleaned/character_anime_works_clean.csv')

Riepilogo Dataset Pulito
Righe originali      :    236,816
Righe dopo cleaning  :    236,814
Righe rimosse totali :          2

Dtypes finali:
anime_mal_id        int64
character_mal_id    int64
character_name        str
role                  str
dtype: object

Valori mancanti residui:
anime_mal_id        0
character_mal_id    0
character_name      0
role                0
dtype: int64

Salvato: datasets_cleaned/character_anime_works_clean.csv
